# Etapa 4 — Campaña Adaptativa Escalonada y Corridas Largas Profundas

Replica simultáneamente las dos estrategias de convergencia profunda de `pinnV3` usando **solo los modelos prometedores** seleccionados previamente:

1. **Subcampaña A (`adaptive_all_cases_campaign.py`)**:
   - Bloques progresivos `ADAM_BLOCKS = (500, 500, 500)` con comprobación del umbral **`RELAXED = 2 × STRICT`**: en cuanto se alcanza, salta anticipadamente a refinamiento **L-BFGS**.
   - Bloques `LBFGS_BLOCKS = (50, 50, 50, 50)` verificando **`STRICT`** (`error_Tmax_abs_K < 0.05`, `nrmse_pct < 0.25`, `balance_pct < 0.5`) tras cada bloque con reanudación exacta (`resume_from`).
2. **Subcampaña B (`long_v2_focused_campaign.py`)**:
   - Corrida larga profunda (`8000 Adam + 20 L-BFGS` en `full_campaign`) con `AccuracyStop(error_Tmax_abs_K=0.01, nrmse_pct=0.1, balance_pct=0.5, every=100)` sobre los 1–2 finalistas.

In [1]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "demo_fast"

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(8, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='demo_fast' | USE_PARALLEL=True (workers=8, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import AdaptiveRunTask, ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
def passes_gate(metrics: dict, gate: dict) -> bool:
    return all(
        metrics.get(k) is not None and np.isfinite(metrics[k]) and float(metrics[k]) < float(lim)
        for k, lim in gate.items()
    )

promo_file = CAMP_WS / "promoted_models.json"
promo = json.loads(promo_file.read_text(encoding="utf-8-sig")) if promo_file.is_file() else {}
PROMOTED = promo.get("promoted_models", ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"])
FINALISTS = promo.get("long_run_finalists", PROMOTED[:2])

if RUN_MODE == "demo_fast":
    ADAM_BLOCKS = (60, 60)
    LBFGS_BLOCKS = (10, 10)
    SEEDS_S4 = (0,)
    CASES_S4A = ("mms_constant", "cable_disk")
    MODELS_S4A = PROMOTED[:2]
    LONG_ADAM, LONG_LBFGS = 120, 15
else:
    ADAM_BLOCKS = (500, 500, 500)
    LBFGS_BLOCKS = (50, 50, 50, 50)
    SEEDS_S4 = (0, 1, 2)
    CASES_S4A = tuple(e["id"] for e in catalog.list_entries())
    MODELS_S4A = PROMOTED
    LONG_ADAM, LONG_LBFGS = 8000, 20

## Subcampaña A: Bloques Adaptativos Reanudables (`adam_i -> lbfgs_j`)

In [3]:
adapt_dir = CAMP_WS / "stage4_adaptive_campaign"
adapt_dir.mkdir(parents=True, exist_ok=True)
adapt_summary_path = adapt_dir / "summary.json"
adapt_summary = json.loads(adapt_summary_path.read_text(encoding="utf-8-sig")) if adapt_summary_path.is_file() else {}
adapt_tasks = []

for case_id in CASES_S4A:
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in MODELS_S4A:
        for seed in SEEDS_S4:
            key = f"{case_id}:{arch}:seed{seed}"
            run_root = adapt_dir / case_id / arch / f"seed_{seed}"
            adapt_tasks.append(
                AdaptiveRunTask(
                    name=key,
                    case_id=case_id,
                    problem_source=prob.doc,
                    architecture=arch,
                    seed=seed,
                    run_root=run_root,
                    adam_blocks=ADAM_BLOCKS,
                    lbfgs_blocks=LBFGS_BLOCKS,
                    relaxed_criteria=RELAXED_CRITERIA,
                    strict_criteria=STRICT_CRITERIA,
                    base_config_kwargs={
                        "lr": 0.002,
                        "width": 32,
                        "depth": 3,
                        "weights": {"pde": 1.0, "energy": 100.0},
                        "sampling": "mixed",
                        "focused": True,
                        "focus_surfaces": False,
                        "resample_every": 100,
                        "dtype": "float32",
                        "device": DEVICE,
                    },
                    reference=ref_vals,
                    reference_label=ref_lbl,
                    overwrite=True,
                )
            )

def _on_stage4a_done(res: dict) -> None:
    key = res["name"]
    rec = {k: v for k, v in res.items() if k != "name"}
    adapt_summary[key] = rec
    write_json_atomic(adapt_summary_path, adapt_summary)
    stages = rec.get("stages") or []
    last_nrmse = stages[-1]["metrics"]["nrmse_pct"] if stages else None
    print(f"[Etapa 4A] {key:32s} | dev={DEVICE} | bloques={len(stages)} | strict={rec.get('strict_converged')} | NRMSE={last_nrmse}")

runner_s4a = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
runner_s4a.run_adaptive(adapt_tasks, on_complete=_on_stage4a_done)

[Etapa 4A] mms_constant:M1_EXACT:seed0      | dev=cpu | bloques=4 | strict=False | NRMSE=0.012359151579963916
[Etapa 4A] mms_constant:M1:seed0            | dev=cpu | bloques=4 | strict=False | NRMSE=4.466991818037012
[Etapa 4A] cable_disk:M1:seed0              | dev=cpu | bloques=4 | strict=False | NRMSE=27.030322609912087
[Etapa 4A] cable_disk:M1_EXACT:seed0        | dev=cpu | bloques=4 | strict=False | NRMSE=27.081065816730597


[{'name': 'mms_constant:M1:seed0',
  'case': 'mms_constant',
  'architecture': 'M1',
  'seed': 0,
  'status': 'completed',
  'relaxed_reached': False,
  'strict_converged': False,
  'stages': [{'name': 'adam_1',
    'steps': 60,
    'metrics': {'error_Tmax_abs_K': 24.89439763810735,
     'nrmse_pct': 41.550673356470604,
     'balance_pct': 84.32655878390231,
     'Tmax_pred_C': 25.103456497192383,
     'elapsed_s': 5.428629400092177},
    'status': 'completed'},
   {'name': 'adam_2',
    'steps': 60,
    'metrics': {'error_Tmax_abs_K': 24.89439763810735,
     'nrmse_pct': 41.550673356470604,
     'balance_pct': 84.32655878390231,
     'Tmax_pred_C': 25.103456497192383,
     'elapsed_s': 0.1406546999933198},
    'status': 'completed'},
   {'name': 'lbfgs_1',
    'steps': 10,
    'metrics': {'error_Tmax_abs_K': 2.4662379243622325,
     'nrmse_pct': 4.466991818037012,
     'balance_pct': 0.009357746073618877,
     'Tmax_pred_C': 47.5316162109375,
     'elapsed_s': 9.662219599937089},
    

## Subcampaña B: Corridas Largas Profundas con Parada Temprana (`AccuracyStop`)

In [4]:
long_dir = CAMP_WS / "stage4_long_focused"
long_dir.mkdir(parents=True, exist_ok=True)
long_summary = {}
stop_cfg = {"error_Tmax_abs_K": 0.01, "nrmse_pct": 0.1, "balance_pct": 0.5, "every": 100, "after_phase": "adam"}
long_tasks = []

for case_id in ("mms_sine", "cable_disk"):
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in FINALISTS[:2]:
        key = f"{case_id}:{arch}"
        out_dir = long_dir / case_id / arch
        long_tasks.append(
            RunTask(
                name=key,
                problem_source=prob.doc,
                architecture=arch,
                output_dir=out_dir,
                config_kwargs={
                    "adam_steps": LONG_ADAM,
                    "lbfgs_steps": LONG_LBFGS,
                    "lr": 0.002,
                    "width": 32,
                    "depth": 3,
                    "weights": {"pde": 1.0, "energy": 100.0},
                    "sampling": "mixed",
                    "focused": True,
                    "focus_surfaces": False,
                    "resample_every": 100,
                    "accuracy_stop": stop_cfg if ref_vals is not None else None,
                    "dtype": "float32",
                    "device": DEVICE,
                },
                reference=ref_vals,
                reference_label=ref_lbl,
                overwrite=True,
            )
        )

def _on_stage4b_done(res: dict) -> None:
    key = res["name"]
    case_id, arch = key.split(":", 1)
    long_summary[key] = {
        "case": case_id,
        "architecture": arch,
        "status": res.get("status"),
        "accuracy_stop": res.get("accuracy_stop"),
        **extract_metrics(res),
    }
    if res.get("error"):
        long_summary[key]["error"] = res.get("error")
    write_json_atomic(long_dir / "summary.json", long_summary)
    print(f"[Etapa 4B] {key:24s} | dev={res.get('device')} | NRMSE={long_summary[key]['nrmse_pct']} | dTmax={long_summary[key]['error_Tmax_abs_K']} K")

runner_s4b = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
runner_s4b.run(long_tasks, on_complete=_on_stage4b_done)
write_json_atomic(long_dir / "summary.json", long_summary)

[Etapa 4B] mms_sine:M1              | dev=cpu | NRMSE=1.7664189553216987 | dTmax=0.24687375810246692 K
[Etapa 4B] mms_sine:M1_EXACT        | dev=cpu | NRMSE=0.009969225754870255 | dTmax=0.0058194181670643275 K
[Etapa 4B] cable_disk:M1            | dev=cpu | NRMSE=27.042448685099565 | dTmax=118.13183208025163 K
[Etapa 4B] cable_disk:M1_EXACT      | dev=cpu | NRMSE=27.106433181503103 | dTmax=118.2868194154567 K


WindowsPath('C:/usr/ths_mia_fiis/tesis_MIA_2026-2_pinn/notebooks_campaign/campaign_workspace/stage4_long_focused/summary.json')